# Day 9 lab: Delta Lake

**Goal:** Explain how Delta Lake turns a folder of Parquet files into a transactional table, and what each DML and maintenance command writes to the data folder and to the log.

**Time:** about 2 h 30 min. Checklist: 7.1–7.10.

Put [`lab_utils.py`](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/lab_utils.py) next to this notebook, run the setup cell, then work through the tasks in order. Write your code in the empty cell under each task and your observations in the notes cell. Keep the Spark UI open (the setup cell prints its address).

In [ ]:
from lab_utils import *

spark = get_spark()

## 📘 Code along

Type the code yourself instead of pasting it, run every cell, and keep the Spark UI open. Swap in `spark = get_spark()` from the starter kit wherever an example builds its own session.

### Task 1. The Delta Lake quickstart — 25 min

Follow the Python parts of Create a table, Read data, Update table data (overwrite, and conditional update without overwrite: update, delete, merge) and Read older versions of data using time travel, side by side with the official `quickstart.py`. Start your session with `get_spark()`: the example's own builder lacks the Delta jar unless you launch the file with `spark-submit --packages io.delta:delta-spark_4.2_2.13:4.4.1 quickstart.py`. Use a path in your lab folder instead of `/tmp`, and after every step list `_delta_log/` and open the newest JSON commit.

[The Delta Lake quickstart](https://docs.delta.io/latest/quick-start.html#create-a-table) · [quickstart.py on GitHub](https://github.com/delta-io/delta/blob/v4.4.1/examples/python/quickstart.py)

In [ ]:
# Task 1: your code here


*Task 1 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 2. Delta utilities and change data feed examples — 20 min

Type the steps of the official `utilities.py` (convert a Parquet folder to Delta, `history()`, `detail()`, `vacuum()`) and of `change_data_feed.py` (a table created with `delta.enableChangeDataFeed = true`, then batch reads of the feed with `readChangeFeed` and `startingVersion`), using paths in your lab folder. Watch the `_change_type` values appear for inserts, updates and deletes.

[Delta utilities and change data feed examples](https://github.com/delta-io/delta/blob/v4.4.1/examples/python/utilities.py) · [change_data_feed.py on GitHub](https://github.com/delta-io/delta/blob/v4.4.1/examples/python/change_data_feed.py)

In [ ]:
# Task 2: your code here


*Task 2 notes: what you predicted, what the plan and the Spark UI showed.*

## ✍️ Exercises

Write these from scratch in the cells below, using the patterns you just typed. Each one ends with a Check you can verify yourself; open the reference solution only after you have a working answer.

### Task 3. Exercise 1: an idempotent upsert — 35 min

Write 100,000 orders to `delta/orders_merge`. Build a source with new amounts for 5 existing order_ids and 5 new ids, and upsert it: `DeltaTable.forPath(spark, "delta/orders_merge").alias("t").merge(src.alias("s"), "t.order_id = s.order_id").whenMatchedUpdateAll(condition="t.amount <> s.amount").whenNotMatchedInsertAll().execute()`. Read `table.history(1)` after each run. Run the same MERGE again: thanks to the condition nothing changes. Then add a duplicate order_id to the source, read the `DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE` error, and fix it by keeping the latest row per key with a `row_number` window. 

**Check:** the first MERGE updates 5 and inserts 5 rows, the second updates 0 and inserts 0. Proves 7.7, 7.2.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day09.py)

In [ ]:
# Task 3: your code here


*Task 3 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 4. Exercise 2: compaction vs Z-ordering — 30 min

Build two copies by appending 50 batches of `make_orders(spark, 20_000, seed=100 + i)` to `delta/a` and to `delta/b` (200 small files each). Set `spark.databricks.delta.optimize.maxFileSize` to `str(8 * 1024 * 1024)` (bytes) so compaction writes several files. Compact a with `DeltaTable.forPath(spark, "delta/a").optimize().executeCompaction()` and Z-order b with `...optimize().executeZOrderBy("customer_id")`; read the returned metrics (about 200 files removed, 7–8 added), the new commit (`dataChange: false`) and the old Parquet files still on disk. Filter `customer_id = 42` on both and compare `len(df.inputFiles())` and "number of files read" in the SQL tab: a reads all 8 files, b reads 1 of 7. Proves 7.4, 7.5.

[Reference solution](https://github.com/usmanov-saidakbar/python_deepdive/blob/main/public/spark-lab/solutions/day09.py)

In [ ]:
# Task 4: your code here


*Task 4 notes: what you predicted, what the plan and the Spark UI showed.*

## 🔬 Prove it

Short, focused checks of today's checklist questions in the plan and the Spark UI.

### Task 5. What UPDATE, DELETE and a bad append write — 20 min

On `delta/orders_merge`, run `table.update(...)` and `table.delete(...)` (the `DeltaTable` API accepts a relative path; SQL on a path needs an absolute one). After each, open the newest commit file in `_delta_log/`: a `remove` of the old file with `dataChange: true` and an `add` of the rewritten one, and `history()` shows the operation and its metrics; the old Parquet files stay in the folder. Then append a DataFrame with an extra column: read the `DELTA_METADATA_MISMATCH` error, and retry with `option("mergeSchema", "true")`. Proves 7.1, 7.6, 7.3.

In [ ]:
# Task 5: your code here


*Task 5 notes: what you predicted, what the plan and the Spark UI showed.*

### Task 6. VACUUM breaks time travel — 20 min

`table.vacuum()` removes nothing yet: files must be older than 7 days. Set `spark.databricks.delta.retentionDurationCheck.enabled` to false (practice data only) and run `table.vacuum(0)`, count the Parquet files again, then read version 0 with `.agg(F.sum("amount"))`: it fails with `FAILED_READ_FILE.FILE_NOT_EXIST`. Delta answers `count()` from log statistics, so prove it by reading data, not by counting. Proves 7.9, 7.8.

In [ ]:
# Task 6: your code here


*Task 6 notes: what you predicted, what the plan and the Spark UI showed.*